In [65]:
import sympy as sp
import numpy as np
from pathlib import Path
x = sp.symbols('x')

In [66]:
data_file = 'raw_count_data.tsv'
with Path(data_file).open('r') as f:
    raw = f.read().split('\n')
raw = [[int(_) for _ in row.split('\t')] for row in raw]

In [67]:
row_sums = {_: 0 for _ in range(24)}
M_dict = {(r, c): 0 for r in range(24) for c in range(24)}
V_dict = {r: 0 for r in range(24)}
for row in raw:
    pre, post, runs, freq = row
    row_sums[pre] += freq
    if post < 24:
        term = -freq * x**runs
        tup = (pre, post)
        M_dict[tup] += term
    else:
        term = freq * x**runs
        V_dict[pre] += term
for _ in range(24):
    tup = (_, _)
    M_dict[tup] += row_sums[_]

In [68]:
M = sp.zeros(8)
V = sp.zeros(8, 1)
for row in range(16, 24):
    V[row - 16] += V_dict[row]
    for col in range(16, 24):
        tup = (row, col)
        M[row - 16, col - 16] += M_dict[tup]
X = sp.simplify(M**-1 * V)

In [69]:
func_dict = {r + 16: X[r] for r in range(8)}
func_dict[24] = 1

In [70]:
M = sp.zeros(8)
V = sp.zeros(8, 1)
for row in range(8, 16):
    V[row - 8] += V_dict[row]
    for col in range(8, 16):
        tup = (row, col)
        M[row - 8, col - 8] += M_dict[tup]
        tup2 = (row, col + 8)
        V[row - 8] -=  M_dict[tup2] * func_dict[col + 8]
X = sp.simplify(M**-1 * V)

In [71]:
for r in range(8):
    func_dict[r + 8] = X[r]


In [72]:
M = sp.zeros(8)
V = sp.zeros(8, 1)
for row in range(8):
    V[row] += V_dict[row]
    for col in range(8):
        tup = (row, col)
        M[row, col] += M_dict[tup]
        tup2 = (row, col + 8)
        V[row] -=  M_dict[tup2] * func_dict[col + 8]
        tup3 = (row, col + 16)
        V[row] -=  M_dict[tup3] * func_dict[col + 16]
X = sp.simplify(M**-1 * V)

In [73]:
for r in range(8):
    func_dict[r] = X[r]

In [76]:
for r in range(24):
    f = func_dict[r]
    fprime = sp.diff(f, x)
    print(r, f.subs(x, 1), fprime.subs(x, 1).evalf())

0 1 0.493621835054765
1 1 0.891122679172806
2 1 1.12544979523288
3 1 1.50856890399325
4 1 1.37273317478526
5 1 1.80227498620828
6 1 1.96800145188714
7 1 2.37296973282464
8 1 0.262268275567088
9 1 0.525671062160504
10 1 0.675713669247728
11 1 0.930208201421173
12 1 0.945163836029844
13 1 1.18619448929871
14 1 1.37071742694759
15 1 1.61418841500365
16 1 0.100995804602795
17 1 0.227604687772211
18 1 0.320246720831974
19 1 0.450545653692119
20 1 0.359410175585112
21 1 0.511050605080731
22 1 0.574092795904055
23 1 0.790333129701083


In [77]:
import pandas as pd

data = []
for r in range(24):
    series_poly = sp.Poly(func_dict[r].series(x, 0, 20).removeO(), x)
    coeffs = series_poly.all_coeffs()                
    numerical_coeffs = [c.evalf() for c in coeffs][::-1]
    prob_vec = np.array(numerical_coeffs)
    normalized = prob_vec / prob_vec.sum()
    data.append(normalized)
cols = [f'p_{r}' for r in range(20)]
df = pd.DataFrame(data, columns=cols)
    

df

,p_0,p_1,p_2,p_3,p_4,p_5,p_6,p_7,p_8,p_9,p_10,p_11,p_12,p_13,p_14,p_15,p_16,p_17,p_18,p_19
0,0.729594603049233,0.145173919506788,0.0690120515472677,0.0317349731271217,0.0141904118072207,0.00607864968474909,0.00252586913090782,0.00102424701029495,0.000407175202960903,0.000159224072526289,6.14072402582994e-5,2.34046072852888e-5,8.83002363130826e-6,3.30197796163530e-6,1.22520429381714e-6,4.51497486949441e-7,1.65364502126726e-7,6.02347191542067e-8,2.18326388168745e-8,7.87815288214609e-9
1,0.583215020661891,0.163463841097565,0.130883189205333,0.0661091570753635,0.0316532271542295,0.0142359060577036,0.00614455459884669,0.00256722400736561,0.00104547133990353,0.000417016921450989,0.000163519429287641,6.32065129770691e-5,2.41361099322459e-5,9.12069845084364e-6,3.41540851743512e-6,1.26882017129330e-6,4.68063923275860e-7,1.71591864601243e-7,6.25548389638174e-8,2.26903845408340e-8
2,0.400131184166442,0.320357469896537,0.142533834033591,0.0744802961650376,0.0352163730185829,0.0157854141880582,0.00678108828332034,0.00282285751191767,0.00114613122644735,0.000456032818219362,0.000178443698478998,6.88519819552237e-5,2.62513246781685e-5,9.90667337360395e-6,3.70535020653564e-6,1.37509470092590e-6,5.06796085780123e-7,1.85636081096373e-7,6.76238953265324e-8,2.45123912447280e-8
3,0.379398104181300,0.213841050213369,0.151254375894148,0.130322245773997,0.0677229148480958,0.0323038511583334,0.0145219444371128,0.00626271432076989,0.00261472678862806,0.00106412197553860,0.000424211313257152,0.000166255564795789,6.42350837429295e-5,2.45190277895870e-5,9.26207685105747e-6,3.46723622558119e-6,1.28770179415035e-6,4.74905217422575e-7,1.74058619362109e-7,6.34404152951378e-8
4,0.178322014837377,0.529074259885875,0.149140276442251,0.0778235028469626,0.0368172487870893,0.0166297865806140,0.00717648720994553,0.00299847299607475,0.00122102907413827,0.000487015494780484,0.000190954386325654,7.38061038015920e-5,2.81817672631608e-5,1.06487986201862e-5,3.98738486458445e-6,1.48122184766232e-6,5.46388206799020e-7,2.00295177663195e-7,7.30152918990022e-8,2.64834928169943e-8
5,0.145478708601305,0.415448703683156,0.170107695014242,0.136438757074654,0.0713762143711390,0.0342899080282201,0.0154757775447167,0.00669403771507919,0.00280136488796566,0.00114223364191567,0.000456056224883032,0.000178967134231583,6.92219767569366e-5,2.64473080026820e-5,9.99858457029601e-6,3.74559696769616e-6,1.39195031774957e-6,5.13636206652365e-7,1.88347008766159e-7,6.86786611415391e-8
6,0.157924657898173,0.262004475531782,0.290897170461136,0.146423138147765,0.0775319992714673,0.0367123255052231,0.0164837811086100,0.00708911649448684,0.00295323035026394,0.00119963689898004,0.000477476909711993,0.000186876843262093,7.21173891161045e-5,2.74994664477629e-5,1.03785446920365e-5,3.88207092617995e-6,1.44073841454094e-6,5.31005163945273e-7,1.94507726334512e-7,7.08566511501896e-8
7,0.135587950604131,0.252519512018042,0.205209667036488,0.146830737637331,0.132349722952375,0.0688693581057233,0.0329379635559748,0.0148241270992288,0.00639881445452738,0.00267334362665343,0.00108854917833799,0.000434131829068693,0.000170202180796189,6.57787710617056e-5,2.51143498060906e-5,9.48892720315157e-6,3.55279412520070e-6,1.31968383689184e-6,4.86767281074472e-7,1.78428007821700e-7
8,0.840165122163485,0.0965653418136660,0.0388632850629598,0.0150900855497171,0.00585304991436860,0.00219598773952223,0.000809281763570407,0.000294066517333800,0.000105674878940507,3.76325176694134e-5,1.33013248066138e-5,4.67179475849096e-6,1.63205770857175e-6,5.67510691574008e-7,1.96545040183994e-7,6.78290399598195e-8,2.33353635235924e-8,8.00589671153033e-9,2.73987186972389e-9,9.35589318704222e-10
9,0.732309185007227,0.113092778224560,0.0915676417996128,0.0379352529840470,0.0154962935161399,0.00600704878035101,0.00227115014276160,0.000841578241766763,0.000307154668853712,0.000110765520817398,3.95587837503239e-5,1.40156533024951e-5,4.93265036457862e-6,1.72616907427884e-6,6.01133876398054e-7,2.08461327741612e-7,7.20239147772652e-8,2.48036702991306e-

In [78]:
df.to_csv('run_distros.csv')

In [80]:
import pickle

with open('func_dict.p','wb') as f:
    pickle.dump(func_dict, f)